# Lesson 6 - Creating an A2A Health Research Agent using Google ADK

In this lesson, you will build a second agent: a Health Research Agent. Unlike the Policy Agent which used PDF documents, this agent will use Google's Agent Development Kit (ADK) and a tool to search the web for health information. You will also see how to easily wrap an ADK agent into an A2A server using the `to_a2a` helper function.

## 6.1. Define the Research Agent

In [1]:
%cd /content/drive/MyDrive/A2A-MCP

/content/drive/MyDrive/A2A-MCP


In [2]:
%%writefile requirements.txt
google-adk[a2a]==1.19.0
a2a-sdk[http-server]==0.3.16
mcp==1.19.0
agent-framework==1.0.0b251120
agent-framework-a2a==1.0.0b251120
langchain-mcp-adapters
langgraph
langchain[openai]
google-auth==2.42.0
httpx==0.28.1
ipython==8.32.0
nest-asyncio==1.6.0
python-dotenv==1.0.1
requests>=2.32.4,<3.0.0
uvicorn==0.34.0

Overwriting requirements.txt


In [3]:
!uv pip install --system -r requirements.txt

Using Python 3.12.13 environment at: /usr
Checked 15 packages in 105ms


You will write the code for `a2a_research_agent.py`:
- **Google ADK**: You will use `LlmAgent` from `google.adk`, which simplifies agent creation.
- **Tools**: You will equip the agent with `google_search` to allow it to fetch external information.
- **A2A Integration**: Instead of manually defining the Executor and RequestHandler as in Lesson 4, you will use `google.adk.a2a.utils.agent_to_a2a.to_a2a` to automatically wrap the ADK agent into an A2A-compliant application.

In [9]:
%%writefile ./a2a_research_agent.py
import os

import uvicorn
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.agents.llm_agent import LlmAgent
from google.adk.tools import google_search

from helpers import setup_env

setup_env()

PORT = int(os.getenv("RESEARCH_AGENT_PORT"))
HOST = os.getenv("AGENT_HOST")


def main() -> None:
    # Create the Agent
    root_agent = LlmAgent(
        model="gemini-2.5-flash-lite",
        name="HealthResearchAgent",
        tools=[google_search],
        description="Provides healthcare information about symptoms, health conditions, treatments, and procedures using up-to-date web resources.",
        instruction="You are a healthcare research agent tasked with providing information about health conditions. Use the google_search tool to find information on the web about options, symptoms, treatments, and procedures. Cite your sources in your responses. Output all of the information you find.",
    )

    # Make the agent A2A-compatible
    a2a_app = to_a2a(root_agent, host=HOST, port=PORT)
    print("Running Health Research Agent")
    uvicorn.run(a2a_app, host=HOST, port=PORT)


if __name__ == "__main__":
    main()
        

Overwriting ./a2a_research_agent.py


## 6.2. Run the Research Agent Server

Now, activate your new Research Agent.
- Open Terminal 2 by running the cell below.
- Type `uv run a2a_research_agent.py`.

In [6]:
import os

from IPython.display import IFrame, Markdown, display

url_template = os.environ.get("DLAI_LOCAL_URL")
if url_template:
    url = url_template.format(port=8888)
    IFrame(f"{url}terminals/2", width=550, height=600)
else:
    display(Markdown("Open a local terminal and run `uv run a2a_research_agent.py`."))

Open a local terminal and run `uv run a2a_research_agent.py`.

In [10]:
uv sync

Resolved 184 packages in 6ms
Checked 180 packages in 414ms
Note: you may need to restart the kernel to use updated packages.


In [11]:
!uv run a2a_research_agent.py

Running Health Research Agent
INFO:     Started server process [26386]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:9998 (Press CTRL+C to quit)
INFO:     ::1:38488 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
2026-04-15 09:38:47,698 - WARNING - runners.py:270 - App name mismatch detected. The runner is configured with app name "HealthResearchAgent", but the root agent was loaded from "/content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/google/adk/agents", which implies app name "agents".
2026-04-15 09:38:47,704 - WARNING - _api_client.py:105 - Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
2026-04-15 09:38:53,853 - INFO - google_llm.py:163 - Sending out request, model: gemini-2.5-flash-lite, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-04-15 09:38:53,854 - INFO - models.py:7039 - AFC is enabled with max remote calls: 10.
2026-04-15 09:38:54,180 - 

## 6.3. Resources

- [Google Agent Development Kit (ADK)](https://google.github.io/adk-docs/)
- [ADK A2A Integration](https://google.github.io/adk-docs/a2a/)